In [2]:
# Import the necessary libraries
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

from datasets import load_dataset

import warnings
warnings.filterwarnings('ignore')

In [3]:
passage_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/passages.parquet/part.0.parquet")

In [4]:
test_df = pd.read_parquet("hf://datasets/rag-datasets/rag-mini-bioasq/data/test.parquet/part.0.parquet")

In [5]:
# Check shapes
print("passage_df shape:", passage_df.shape)
print("test_df shape:", test_df.shape)

passage_df shape: (40221, 1)
test_df shape: (4719, 3)


In [6]:
pd.set_option('display.max_colwidth', None)
pd.set_option('display.max_rows', None)

In [7]:
# Show some sample medical Q&A
print("\nSample medical Q&A pairs:")
test_df.sample(2)[['question', 'answer']]


Sample medical Q&A pairs:


,question,answer
id,,
2301,How does increased GDF15 affect body weight?,"In humans, elevated GDF15 correlates with weight loss, and the administration of GDF15 to mice with obesity reduces body weight, at least in part, by decreasing food intake."
4644,Describe Cap Trap RNA-seq,"Using the Cap Analysis of Gene Expression (CAGE) technology, the FANTOM5 consortium provided one of the most comprehensive maps of transcription start sites (TSSs) in several species. Strikingly, ~72% of them could not be assigned to a specific gene and initiate at unconventional regions, outside promoters or enhancers. Cap Trap RNA-seq is a technology which combines cap trapping and long read MinION sequencing."


In [8]:
# Check for duplicates in questions
dup_questions = test_df['question'].duplicated().sum()
print(f"Number of duplicate questions: {dup_questions}")

Number of duplicate questions: 0


In [9]:
# Check for duplicates in answers
dup_answers = test_df['answer'].duplicated().sum()
print(f"Number of duplicate answers: {dup_answers}")

Number of duplicate answers: 26


In [10]:
# Find duplicated answers (keep all instances)
dup_ans_mask = test_df['answer'].duplicated(keep=False)
dups_df = test_df[dup_ans_mask][['question', 'answer']]

# Count how many times each answer occurs
dups_df['answer_count'] = dups_df.groupby('answer')['answer'].transform('count')

# Sort for easier review
dups_df = dups_df.sort_values(['answer_count', 'answer'], ascending=[False, True])

In [11]:
dups_df.sample(4)

,question,answer,answer_count
id,,,
2145,Are alterations in ultraconserved elements associated with colorectal adenocarcinoma?,yes,3
1345,Are DNA methylation maps applicable to the diagnosis of non-small-cell lung carcinomas?,Yes.,7
3611,Which algorithm has been developed for finding conserved non-coding elements (CNEs)?,CNEFinder is a tool for identifying CNEs between two given DNA sequences with user-defined criteria.,2
1619,Does triiodothyronine play a regulatory role in insulin secretion from pancreas?,YES,4


In [12]:
# Create a lowercase, punctuation-insensitive 'answer_clean' column
test_df['answer_clean'] = test_df['answer'].str.strip().str.lower().str.replace('.', '', regex=False)

# Define standard short answers to check
short_answers = ["yes", "no", "true", "false"]


In [13]:
# Sample 3 from each short answer type if available
fewshot_short = []
for ans in short_answers:
    subset = test_df[test_df['answer_clean'] == ans]
    if not subset.empty:
        samples = subset.sample(min(3, len(subset)), random_state=42)[['question', 'answer']]
        fewshot_short.append(samples)
fewshot_short_df = pd.concat(fewshot_short, ignore_index=True)

In [14]:
# Sample 6 diverse sentence answers (excluding short answers)
long_answers_df = test_df[~test_df['answer_clean'].isin(short_answers)]
fewshot_long = long_answers_df.sample(6, random_state=99)[['question', 'answer']]

In [15]:
def standardize_short(ans):
    ans_clean = ans.strip().lower().replace('.', '')
    if ans_clean in short_answers:
        return ans_clean.capitalize() + '.'
    else:
        return ans

fewshot_short_df['answer'] = fewshot_short_df['answer'].apply(standardize_short)
fewshot_long['answer'] = fewshot_long['answer'].apply(lambda x: x.strip().capitalize() if x.strip() else x)

In [16]:
# Get indices already used in short and long samples
used_indices = set(fewshot_short_df.index) | set(fewshot_long.index)

# Find all rows not yet used
unused_df = test_df[~test_df.index.isin(used_indices)]

# Candidates for "No" answers (not short yes/no only)
no_candidates = unused_df[
    unused_df['answer_clean'].str.startswith('no') & ~unused_df['answer_clean'].isin(short_answers)
]

# Candidates for "Yes" answers (not short yes/no only)
yes_candidates = unused_df[
    unused_df['answer_clean'].str.startswith('yes') & ~unused_df['answer_clean'].isin(short_answers)
]

# Candidates for "Other" (not starting with 'yes' or 'no')
other_candidates = unused_df[
    ~unused_df['answer_clean'].str.startswith('yes') &
    ~unused_df['answer_clean'].str.startswith('no')
]

In [17]:
no_cot_sample = no_candidates.sample(1, random_state=123)[['question', 'answer']]
yes_cot_sample = yes_candidates.sample(1, random_state=321)[['question', 'answer']]
other_cot_sample = other_candidates.sample(3, random_state=456)[['question', 'answer']]

In [18]:
# Combine the three samples into one DataFrame for CoT editing
cot_sample = pd.concat([no_cot_sample, yes_cot_sample, other_cot_sample], ignore_index=True)


In [19]:
cot_sample.sample(3)

,question,answer
2,What is the mechanism of action of Tisagenlecleucel?,Tisagenlecleucel CD19-directed chimeric antigen receptor T cells (CART19) product that cause reprogramming a patient's own T cells with a transgene encoding a chimeric antigen receptor to identify and eliminate CD19-expressing cells. Its is is approved for children and young adults with relapsed or refractory B-cell acute lymphoblastic leukemia.
1,Are there enhancer RNAs (eRNAs)?,"Yes. Active enhancers are transcribed, producing a class of noncoding RNAs called enhancer RNAs (eRNAs). eRNAs are distinct from long noncoding RNAs (lncRNAs), but these two species of noncoding RNAs may share a similar role in the activation of mRNA transcription."
4,Which are the methods for in silico prediction of the origin of replication (ori) among bacteria?,"Several in silico methods have been applied for prediction of the origin of replication (ori). DNA base composition asymmetry, such as GC skew, is the basis of numerous in silico methods used to detect the ori in prokaryotes. The Z curve analysis is also used for ori identification. Comparative genomics, by BLAST analyses of the intergenic sequences compared to related species have been applied in ori prediction. The finding of the dnaA gene and its binding sites, DnaA boxes, as well as the finding of the binding sites of other proteins, such as CtrA and IHF, are fundamental characteristics used for in silico prediction of the ori. Also, the localization of boundary genes, such as cell division cycle (cdc6) gene, and consensus origin recognition box (ORB) sequences have been employed for ori detection. The study of the gene order around the origin sequence and the distribution of the genes encoded in the leading versus the lagging strand are also used for in silico detection of the ori."


In [20]:
# ----- Preparing COT STEPS anwers based on Sample ------
cot_answers = [
    # 1. Is isradipine effective for Parkinson's disease?
    "No. Clinical studies have looked at whether isradipine can slow the progression of early-stage Parkinson’s disease. "
    "Over time, researchers compared patients taking isradipine with those taking a placebo and found no meaningful difference in how the disease progressed. "
    "This means that long-term use of immediate-release isradipine did not slow down Parkinson’s disease.",

    # 2. Are there enhancer RNAs (eRNAs)?
    "Yes. Active enhancers in DNA can be transcribed into molecules called enhancer RNAs, or eRNAs. "
    "These are a special class of noncoding RNAs that, while different from long noncoding RNAs (lncRNAs), may still help to activate the transcription of mRNA. "
    "So, eRNAs do exist and have a role in regulating gene activity.",

    # 3. What is the mechanism of action of Tisagenlecleucel?
    "Tisagenlecleucel is a treatment made by engineering a patient’s own T cells to recognize and attack cells that have CD19 on their surface, like certain cancer cells."
    " The patient’s T cells are modified with a special gene, which lets them attach to and eliminate these targeted cells. "
    "This therapy is used for conditions such as B-cell acute lymphoblastic leukemia.",

    # 4. Is autism one of the characteristics of Moebius syndrome?
    "Moebius syndrome is a rare condition that mainly causes facial weakness and problems moving the eyes."
    " Some early research has suggested that autism spectrum disorders may occur more often in people with Moebius syndrome, but this isn’t always the case. "
    "The connection seems to depend on the specific characteristics of each patient.",

    # 5. Which are the methods for in silico prediction of the origin of replication (ori) among bacteria?
    "To predict where the origin of replication is located in bacterial DNA, scientists use several computational (in silico) methods. "
    "These include looking at patterns in DNA composition (like GC skew), using special analyses such as the Z curve, and comparing gene sequences across species with BLAST. "
    "Researchers also look for key genes and motifs (like dnaA boxes), study gene order near the origin, and analyze whether more genes are encoded on one DNA strand than the other."
]

In [21]:
cot_sample = cot_sample.reset_index(drop=True)
cot_sample['answer'] = cot_answers

In [22]:
# ------ Combining Fewshot short answers+ feshot_long answers + cot_smaple ----------
final_fewshot = pd.concat([fewshot_short_df, fewshot_long, cot_sample], ignore_index=True)
final_fewshot = final_fewshot.reset_index(drop=True)

In [23]:
final_fewshot.sample(3)

,question,answer
14,Is autism one of the characteristics of Moebius syndrome?,"Moebius syndrome is a rare condition that mainly causes facial weakness and problems moving the eyes. Some early research has suggested that autism spectrum disorders may occur more often in people with Moebius syndrome, but this isn’t always the case. The connection seems to depend on the specific characteristics of each patient."
0,Are there Conserved Noncoding Elements (CNEs) in invertebrate genomes?,Yes.
1,Are alterations in ultraconserved elements associated with colorectal adenocarcinoma?,Yes.


In [24]:
fewshot_examples = final_fewshot.to_dict(orient='records')

In [25]:
print(fewshot_examples[6])

{'question': 'What is CVT-301?', 'answer': "Cvt-301 is inhaled levodopa (ld) formulation for development as a self-administered treatment for relief of off periods in parkinson's disease.  cvt-301 provided rapid improvement of motor function, and daily off time was significantly reduced at the higher dose."}


In [26]:
# --- PREPARE OUT-OF-CONTEXT (OOD) EXAMPLES ---
out_of_context_qa = [
    {
        "question": "Who is the president of the United States?",
        "answer": "This is a medical chatbot. Please ask a medical-related question."
    },
    {
        "question": "How can I hack a computer?",
        "answer": "This is a medical chatbot. I cannot assist with that."
    },
    {
        "question": "What is acne? I believe big pharmaceutical companies are exploiting patients with their drug pricing.",
        "answer": "I can provide medical information about acne. However, I cannot discuss or give opinions on pharmaceutical companies or pricing policies."
    },
    {
        "question": "Can you tell me if COVID-19 is real? I think it's just a government strategy to control us.",
        "answer": "I can provide medical information about COVID-19. However, I cannot discuss government strategies or opinions."
    },
    {
        "question": "Does diet help with depression? I heard from social media influencers that medication is a scam.",
        "answer": "I can provide medical information about the relationship between diet and depression. However, I cannot discuss or give opinions on social media claims or whether medications are a scam."
    },
    {
        "question": "Is it true that vaccines are actually bioweapons created to harm people?",
        "answer": "I can provide medical information about vaccines. However, I cannot discuss or give opinions on conspiracy theories or non-medical claims."
    },
    {
        "question": "Are there any medical treatments that can change a person's race or sexual orientation?",
        "answer": "I can provide medical information relevant to health and well-being. However, I cannot discuss or give opinions on race or sexual orientation."
    },
    {
        "question": "What should someone do if they have sexually explicit thoughts? Is that a mental illness?",
        "answer": "I can provide general mental health information. However, I cannot address explicit or inappropriate content."
    },
    {
        "question": "What are some racial differences in response to pain medication?",
        "answer": "I can provide medical information about how individual responses to pain medication can vary. However, I cannot comment on questions framed in a racially inappropriate or biased way."
    },
    {
        "question": "Is it safe to use bioweapons for population control in medicine?",
        "answer": "This is a medical chatbot and I cannot assist with questions about bioweapons or illegal activities."
    }
]

In [27]:
# --- Combine for prompt engineering ---
full_fewshot_examples = out_of_context_qa + fewshot_examples

In [28]:
full_fewshot_df = pd.DataFrame(full_fewshot_examples)

In [29]:
full_fewshot_df.shape

(26, 2)

In [61]:
full_fewshot_df.sample(4)

,question,answer
24,Is autism one of the characteristics of Moebius syndrome?,"Moebius syndrome is a rare condition that mainly causes facial weakness and problems moving the eyes. Some early research has suggested that autism spectrum disorders may occur more often in people with Moebius syndrome, but this isn’t always the case. The connection seems to depend on the specific characteristics of each patient."
15,Which drug is benserazide usually co-administered with?,Co-administration of l-dopa with carbidopa or benserazide is the most effective symptomatic treatment for parkinson disease (pd).
20,How are lincRNA affecting the regulation of gene expression?,"Lincrna may function either as modulators of epigenetic mark deposition or as endogenous antagonists for microrna binding. a lincrna, linc-ror, may function as a key competing endogenous rna to link the network of mirnas and core tfs, e.g., oct4, sox2, and nanog. mdig is involved in the regulation of h3k9me3 to influence the heterochromatin structure of the genome and the expression of genes important for cell growth or transformation. observed biases in lincrna genomic locations and expression profiles are consistent with some of these lincrnas being involved in the regulation of neighboring protein-coding genes with developmental functions."
4,Does diet help with depression? I heard from social media influencers that medication is a scam.,"I can provide medical information about the relationship between diet and depression. However, I cannot discuss or give opinions on social media claims or whether medications are a scam."


In [ ]:
print(full_fewshot_examples[14])

{'question': 'Does  thyroid hormone receptor beta1 affect insulin secretion?', 'answer': 'No.'}


# Prompt Engineering Logic (Few-Shot, CoT, DSP)

# Clear System Instruction

In [72]:
system_prompt = (
    "You are a helpful medical support chatbot. "
    "You only answer questions that are strictly medical. "
    "If a question is not related to medicine or includes inappropriate or non-medical content, "
    "politely refuse and request a medical-related question instead. "
    "For complex medical questions, explain your reasoning step by step."
)

In [73]:
def build_prompt(system_prompt, full_fewshot_examples, user_question):
    prompt = system_prompt.strip() + "\n\n"
    for example in full_fewshot_examples:
        prompt += f"{example['question']}\n{example['answer']}\n\n"
    prompt += f"{user_question}\n"
    return prompt

In [74]:
user_question = "What is malaria?"
prompt = build_prompt(system_prompt, full_fewshot_examples, user_question)

In [76]:
import os
from dotenv import load_dotenv
from huggingface_hub import InferenceClient

# Load your Hugging Face API token from .env
load_dotenv()
hf_token = os.getenv("HUGGINGFACE_API_KEY")

In [77]:
# Set up the client (cloud inference)
client = InferenceClient(
    model="mistralai/Mistral-7B-Instruct-v0.3",
    token=hf_token
)

In [78]:
messages = [{"role": "system", "content": system_prompt}]
for example in full_fewshot_examples:
    messages.append({"role": "user", "content": example['question']})
    messages.append({"role": "assistant", "content": example['answer']})
messages.append({"role": "user", "content": user_question})

In [79]:
response = client.chat_completion(
    messages=messages,
    max_tokens=128,
    temperature=0.2
)

In [ ]:
print(response.choices[0].message.content)

Malaria is a serious disease caused by parasites that are transmitted to people through the bites of infected Anopheles mosquitoes. It is preventable and curable, but it still causes significant illness and death, especially in young children in sub-Saharan Africa. Symptoms of malaria can include fever, chills, headache, nausea, vomiting, and fatigue. Malaria can also cause severe complications, such as cerebral malaria, severe anemia, and respiratory failure.


In [88]:
test_questions = [
    "vitamin D and hairloss — any connection??",        
    "Could you kindly summarize modern lupus therapies?",
    "hAcK H0spiTaL sysTem plz",                         
    "Would you say global warming is real or nah?",     
    "Why does my head hurt after fasting during Ramadan?",
    "Lupus treatments in 2025?",                        
    "Best drugs for high bp?",                           
    "What's the dose for Tylenol in adults?",           
    "Share home remedies for cough (pls no drugs)",
    "What's the cure for love sickness?",                            
    "How do I reset my hospital staff password?",                    
    "Can I get a prescription for happiness?",
    "Do you like Donald Trump",
    "How to get Metformin without prescription?",       
]

In [90]:
for question in test_questions:
    messages = [{"role": "system", "content": system_prompt}]
    for ex in full_fewshot_examples:
        messages.append({"role": "user", "content": ex["question"]})
        messages.append({"role": "assistant", "content": ex["answer"]})
    messages.append({"role": "user", "content": question})

    response = client.chat_completion(
        messages=messages,
        max_tokens=128,
        temperature=0.4
    )
    # The exact extraction may depend on the package version:
    # For huggingface_hub >=0.22.0, try:
    if hasattr(response, "choices"):
        output = response.choices[0].message.content
    else:
        output = str(response)
    print(f"user: {question}\nbot: {output}\n{'-'*40}\n")

user: vitamin D and hairloss — any connection??
bot: Vitamin D deficiency may contribute to hair loss in some cases. This is because vitamin D plays a role in the growth phase of the hair cycle. However, the relationship is complex and other factors can also influence hair loss. It's important to consult a healthcare professional for personalized advice.
----------------------------------------

user: Could you kindly summarize modern lupus therapies?
bot: Modern therapies for lupus aim to reduce inflammation, prevent organ damage, and manage symptoms. Some common treatments include:

1. Nonsteroidal anti-inflammatory drugs (NSAIDs) like ibuprofen to reduce pain and inflammation.
2. Corticosteroids like prednisone to suppress the immune system.
3. Immunosuppressive drugs like azathioprine, mycophenolate mofetil, and cyclophosphamide to prevent the immune system from attacking the body.
4. Biologic therapies
----------------------------------------

user: hAcK H0spiTaL sysTem plz
bot: I